In [52]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

sales = pd.read_csv(PROCESSED_DIR / "fact_sales_daily.csv", parse_dates=["Date"])
inv = pd.read_csv(PROCESSED_DIR / "fact_inventory.csv", parse_dates=["Snapshot_Date"])
forecast = pd.read_csv(PROCESSED_DIR / "forecast_demand_30d.csv", parse_dates=["Date"])

print(latest_date, latest_inv.shape, forecast.shape)

2025-12-01 00:00:00 (50, 19) (1500, 8)


# Demand statistics

Historical daily demand variability (drives Safety Stock) and forward-looking average daily demand from the trained model (drives Reorder Point) — not the raw `Safety_Stock`/`Reorder_Point` columns from the source data.

In [55]:
demand_std = sales.groupby("SKU")["Units_Sold"].std()
avg_daily_demand = forecast.groupby("SKU")["Forecast_Units"].mean()
lead_times = latest_inv["Lead_Time_Days"]

print(demand_std.describe())

count    50.000000
mean      4.401532
std       1.527029
min       1.696323
25%       3.213060
50%       4.529439
75%       5.881037
max       7.036270
Name: Units_Sold, dtype: float64


# Calculated Safety Stock & Reorder Point

`Safety_Stock = Z × demand_std × √lead_time` (Z=1.65 → ~95% service level)

`Reorder_Point = avg_daily_demand × lead_time + Safety_Stock`

In [58]:
Z = 1.65  # ~95% service level; use 2.33 for ~99%

calc_safety_stock = Z * demand_std * np.sqrt(lead_times)
calc_reorder_point = (avg_daily_demand * lead_times) + calc_safety_stock

calc_safety_stock.describe()

count    50.000000
mean     19.797287
std       8.632312
min       4.963414
25%      13.019199
50%      18.018921
75%      26.141180
max      40.008086
dtype: float64

# Expected demand over lead time (from the forecast)

In [61]:
expected_demand_lt = {}
for sku, lt in lead_times.items():
    sku_fc = forecast[forecast["SKU"] == sku].sort_values("Date")
    expected_demand_lt[sku] = sku_fc["Forecast_Units"].head(lt).sum()
expected_demand_lt = pd.Series(expected_demand_lt)

expected_demand_lt.head()

SKU001     55.90
SKU002     26.96
SKU003     15.27
SKU004     29.71
SKU005    101.40
dtype: float64

# Project stock position forward

In [64]:
projected_stock = latest_inv["Current_Stock"] + latest_inv["On_Order"] - expected_demand_lt
print("SKUs with negative projected stock:", (projected_stock < 0).sum())
projected_stock.head()

SKUs with negative projected stock: 5


SKU
SKU001    862.10
SKU002    528.04
SKU003    345.73
SKU004    483.29
SKU005    660.60
dtype: float64

# Assemble the output table

In [67]:
out = pd.DataFrame({
    "SKU": latest_inv.index,
    "Product_Name": latest_inv["Product_Name"],
    "Category": latest_inv["Category"],
    "Subcategory": latest_inv["Subcategory"],
    "Current_Stock": latest_inv["Current_Stock"],
    "On_Order": latest_inv["On_Order"],
    "Lead_Time_Days": lead_times,
    "Avg_Daily_Demand": avg_daily_demand.reindex(latest_inv.index).round(2),
    "Demand_Std": demand_std.reindex(latest_inv.index).round(2),
    "Calculated_Safety_Stock": calc_safety_stock.reindex(latest_inv.index).round(0),
    "Calculated_Reorder_Point": calc_reorder_point.reindex(latest_inv.index).round(0),
    "Expected_Demand_LeadTime": expected_demand_lt.reindex(latest_inv.index).round(2),
    "Projected_Stock": projected_stock.reindex(latest_inv.index).round(2),
    "Selling_Price": latest_inv["Selling_Price"],
}).reset_index(drop=True)

out.head()

,SKU,Product_Name,Category,Subcategory,Current_Stock,On_Order,Lead_Time_Days,Avg_Daily_Demand,Demand_Std,Calculated_Safety_Stock,Calculated_Reorder_Point,Expected_Demand_LeadTime,Projected_Stock,Selling_Price
0,SKU001,Product 001,Furniture,Chair,771,147,4,13.64,4.75,16.0,70.0,55.90,862.10,3664.05
1,SKU002,Product 002,Home Decor,Table,526,29,3,9.49,3.71,11.0,39.0,26.96,528.04,3805.69
2,SKU003,Product 003,Kitchen,Cushion,349,12,3,5.36,2.76,8.0,24.0,15.27,345.73,8078.23
3,SKU004,Product 004,Lighting,Cookware,259,254,6,4.76,2.36,10.0,38.0,29.71,483.29,6861.57
4,SKU005,Product 005,Storage,Lamp,471,291,6,16.82,4.81,19.0,120.0,101.40,660.60,9493.22


# Stockout Risk status: 
- Critical
- Warning
- Overstock
- Normal

`Normal` is added as a 4th bucket for SKUs that are neither at risk nor overstocked — without it, every healthy SKU would be forced into Warning or Overstock, misrepresenting them. 

In [70]:
def classify(row):
    if row["Projected_Stock"] < 0:
        return "Critical"
    elif row["Projected_Stock"] <= row["Calculated_Reorder_Point"]:
        return "Warning"
    elif row["Projected_Stock"] > row["Calculated_Reorder_Point"] * 2:
        return "Overstock"
    else:
        return "Normal"

out["Stockout_Risk_Status"] = out.apply(classify, axis=1)
out["Stockout_Risk_Status"].value_counts()

Stockout_Risk_Status
Overstock    31
Warning       9
Critical      5
Normal        5
Name: count, dtype: int64

In [72]:
# NOTE: Overstock is the largest bucket here — driven by the Reorder_Point x 2
# cutoff, which is a modeling choice, not derived from the data. Worth tuning
# against what the business actually considers "too much stock" before trusting
# this as final.
out[out["Stockout_Risk_Status"] == "Overstock"].shape[0], out.shape[0]

(31, 50)

# Projected Lost Revenue

Only Critical SKUs have a stockout gap; lost units = shortfall = `-Projected_Stock` (clipped at 0).

In [75]:
out["Lost_Units"] = np.where(
    out["Stockout_Risk_Status"] == "Critical",
    (-out["Projected_Stock"]).clip(lower=0), 0
)
out["Projected_Lost_Revenue"] = (out["Lost_Units"] * out["Selling_Price"]).round(2)

out = out.sort_values("Projected_Lost_Revenue", ascending=False)

print(f"Total Projected Lost Revenue (Critical SKUs): {out['Projected_Lost_Revenue'].sum():,.2f}")
out[out["Stockout_Risk_Status"] == "Critical"][
    ["SKU", "Current_Stock", "On_Order", "Expected_Demand_LeadTime",
     "Projected_Stock", "Calculated_Safety_Stock", "Lost_Units", "Projected_Lost_Revenue"]
]

Total Projected Lost Revenue (Critical SKUs): 3,166,062.52


,SKU,Current_Stock,On_Order,Expected_Demand_LeadTime,Projected_Stock,Calculated_Safety_Stock,Lost_Units,Projected_Lost_Revenue
11,SKU012,47,60,304.36,-197.36,40.0,197.36,1732696.46
30,SKU031,93,40,236.90,-103.90,32.0,103.90,905585.13
16,SKU017,95,26,168.26,-47.26,26.0,47.26,400638.14
39,SKU040,67,29,129.01,-33.01,23.0,33.01,80892.99
9,SKU010,83,70,222.71,-69.71,31.0,69.71,46249.80


# Validate against known findings

SKU012 was the deepest stockout risk in earlier analysis — confirm it still shows up as Critical with the corrected, calculated thresholds.

In [78]:
out[out["SKU"] == "SKU012"][
    ["SKU", "Calculated_Safety_Stock", "Calculated_Reorder_Point",
     "Projected_Stock", "Stockout_Risk_Status", "Projected_Lost_Revenue"]
]

,SKU,Calculated_Safety_Stock,Calculated_Reorder_Point,Projected_Stock,Stockout_Risk_Status,Projected_Lost_Revenue
11,SKU012,40.0,344.0,-197.36,Critical,1732696.46


In [83]:
out.to_csv(PROCESSED_DIR / "inventory_risk_optimization.csv", index=False)
print("Saved", out.shape, "to", PROCESSED_DIR / "inventory_risk_optimization.csv")
out.head(10)

Saved (50, 17) to C:\Users\rocky\Desktop\ForeSight\data\processed\inventory_risk_optimization.csv


,SKU,Product_Name,Category,Subcategory,Current_Stock,On_Order,Lead_Time_Days,Avg_Daily_Demand,Demand_Std,Calculated_Safety_Stock,Calculated_Reorder_Point,Expected_Demand_LeadTime,Projected_Stock,Selling_Price,Stockout_Risk_Status,Lost_Units,Projected_Lost_Revenue
11,SKU012,Product 012,Home Decor,Table,47,60,12,25.31,7.00,40.0,344.0,304.36,-197.36,8779.37,Critical,197.36,1732696.46
30,SKU031,Product 031,Furniture,Chair,93,40,13,18.13,5.31,32.0,267.0,236.90,-103.90,8715.93,Critical,103.90,905585.13
16,SKU017,Product 017,Home Decor,Cabinet,95,26,12,14.06,4.47,26.0,194.0,168.26,-47.26,8477.32,Critical,47.26,400638.14
39,SKU040,Product 040,Storage,Sofa,67,29,9,14.51,4.72,23.0,154.0,129.01,-33.01,2450.56,Critical,33.01,80892.99
9,SKU010,Product 010,Storage,Sofa,83,70,13,16.98,5.25,31.0,252.0,222.71,-69.71,663.46,Critical,69.71,46249.80
0,SKU001,Product 001,Furniture,Chair,771,147,4,13.64,4.75,16.0,70.0,55.90,862.10,3664.05,Overstock,0.00,0.00
5,SKU006,Product 006,Furniture,Shelf,359,485,13,6.82,2.81,17.0,105.0,86.13,757.87,8106.04,Overstock,0.00,0.00
6,SKU007,Product 007,Home Decor,Cabinet,207,460,8,24.30,6.53,30.0,225.0,189.60,477.40,5114.09,Overstock,0.00,0.00
1,SKU002,Product 002,Home Decor,Table,526,29,3,9.49,3.71,11.0,39.0,26.96,528.04,3805.69,Overstock,0.00,0.00
2,SKU003,Product 003,Kitchen,Cushion,349,12,3,5.36,2.76,8.0,24.0,15.27,345.73,8078.23,Overstock,0.00,0.00
